In [ ]:
!cp /content/drive/MyDrive/School/University-of-Guelph/MDS/Course/26-2S_DATA6700/1_data/data_siamese_unet.zip /content/

In [ ]:
!unzip -q /content/data_siamese_unet.zip -d /content/data

In [ ]:
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
from torch.utils.data import DataLoader, Dataset

In [ ]:
DIR_DATA_DRIVE = Path("data")
DIR_DATA_LOCAL = Path("/content/data")

NORMALIZE_METADATA = True
TARGET_NAME = "zscore"

# ============================================================
# Directories
# ============================================================
DIR_DATA = DIR_DATA_LOCAL

DIR_ICEYE = DIR_DATA / "1_ICEYE"
DIR_TERRAIN = DIR_DATA / "3_Terrain" / "depmap"
DIR_ZSCORE = DIR_DATA / "4_ICEYE_Zscore"

FILEPATH_PAIR_MANIFEST = DIR_DATA / "0_metadata" / "pair-manifest-siamese-unet.csv"

DIR_RESULTS = DIR_DATA_DRIVE / "3_results" / "siamese-unet"
# ============================================================
# Input Parameters
# ============================================================
LOOK_SIDE = {
    "Left": 0,
    "Right": 1
}
ORBIT_DIRECTION = {
    "Ascending": 0,
    "Descending": 1
}
METADATA_COLUMNS = [
    "delta_days",
    "rain_1d",
    "rain_3d",
    "rain_5d",
    "incidence_angle",
]

# ============================================================
# Dataset
# ============================================================
PATCH_SIZE = 256

# ============================================================
# Model
# ============================================================
INPUT_CHANNELS = 2
METADATA_DIM = len(METADATA_COLUMNS) + 2

# ============================================================
# DataLoader
# ============================================================
BATCH_SIZE = 4
NUM_WORKERS = 2
PIN_MEMORY = True
SHUFFLE_TRAIN = True
SHUFFLE_VALID = False

# ============================================================
# Training
# ============================================================
EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================================
# Misc
# ============================================================
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


In [ ]:
# if not DIR_DATA_LOCAL.exists():
#     print("Copying dataset to local disk...")
#     shutil.copytree(DIR_DATA_DRIVE, DIR_DATA_LOCAL)
#     print("Done!")
# else:
#     print("Dataset already exists.")

In [ ]:
class FloodDataset(Dataset):
    def __init__(
        self,
        split: str,
        transform=None
    ):
        """
        split:
            "train", "valid", or "test"
        """
        self.transform = transform
        self.samples = []
        pair_manifest = pd.read_csv(
            DIR_DATA / FILEPATH_PAIR_MANIFEST
        )

        # --------------------------------------------------
        # Filter pairs
        # --------------------------------------------------
        if split not in ("train", "valid", "test"):
            raise ValueError(
                "split must be 'train', 'valid', or 'test'"
            )

        pair_manifest = pair_manifest[
            pair_manifest["split"] == split
        ]

        # --------------------------------------------------
        # Build sample list
        # --------------------------------------------------
        for _, pair in pair_manifest.iterrows():
            sar_base_dir = DIR_ICEYE / pair["sar_base"]
            sar_target_dir = DIR_ICEYE / pair["sar_target"]

            # Every SAR patch becomes one sample
            for sar_path in sorted(sar_base_dir.glob("*.tif")):
                parts = sar_path.stem.split("_", 1)
                if len(parts) != 2:
                    print(f"Unexpected filename: {sar_path.name}")
                    continue
                patch_id = parts[1]
                # Check split
                split_in_patch_id = get_split_from_patch_filename(patch_id)
                if split not in split_in_patch_id:
                    continue
                # Get year
                year = get_year_from_patch_id(patch_id)

                base_iceye_id = pair["sar_base"].split("_")[-1]
                target_iceye_id = pair["sar_target"].split("_")[-1]

                metadata = [pair[col] for col in METADATA_COLUMNS]
                metadata.append(ORBIT_DIRECTION[pair["orbit_direction"]])
                metadata.append(LOOK_SIDE[pair["look_side"]])
                metadata = np.array(
                    metadata,
                    dtype=np.float32
                )

                sample = {
                    "pair_id": pair["pair_id"],
                    "patch_id": patch_id,
                    "sar_base":
                        sar_base_dir /
                        f"{base_iceye_id}_{patch_id}.tif",
                    "sar_target":
                        sar_target_dir /
                        f"{target_iceye_id}_{patch_id}.tif",
                    "terrain":
                        DIR_TERRAIN /
                        f"patches_{year}" /
                        f"depmap_{patch_id}.tif",
                    "zscore":
                        DIR_ZSCORE /
                        pair["sar_target"] /
                        f"z_{target_iceye_id}_{patch_id}.tif",
                    "metadata": metadata
                }
                # Skip incomplete samples
                paths = [
                    sample["sar_base"],
                    sample["sar_target"],
                    sample["terrain"],
                    sample["zscore"],
                ]
                if all(p.exists() for p in paths):
                    self.samples.append(sample)

        print(
            f"{split}: {len(self.samples)} samples"
        )

    def __len__(self):
        return len(self.samples)

    def _read_tiff(self, path):

        with rasterio.open(path) as src:
            img = src.read(1).astype(np.float32)

        return img

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # ----------------------------
        # Load images
        # ----------------------------
        sar_base = self._read_tiff(sample["sar_base"])
        sar_target = self._read_tiff(sample["sar_target"])
        terrain = self._read_tiff(sample["terrain"])
        zscore = self._read_tiff(sample["zscore"])

        # ----------------------------
        # Build 3-channel inputs
        # ----------------------------
        before = np.stack([
            sar_base,
            terrain
        ])
        after = np.stack([
            sar_target,
            terrain
        ])
        before = torch.from_numpy(before)
        after = torch.from_numpy(after)
        metadata = torch.from_numpy(sample["metadata"])
        zscore = torch.from_numpy(zscore).unsqueeze(0)

        if self.transform is not None:
            before = self.transform(before)
            after = self.transform(after)

        return {
            "before": before,
            "after": after,
            "metadata": metadata,
            "zscore": zscore,
            "pair_id": sample["pair_id"],
            "patch_id": sample["patch_id"]
        }

def get_year_from_patch_id(patch_id: str) -> int:
    """
    Extract the year from a patch ID.

    Examples
    --------
    B24BC0120 -> 2024
    W25CB0120 -> 2025
    """

    return 2000 + int(patch_id[2:4])

from pathlib import Path

def get_split_from_patch_filename(patch_id: str) -> str:
    """
    Extract the dataset split from a SAR patch ID.

    Examples
    --------
    TB24CC0120 -> "train/valid"
    EB24CC0120 -> "test"
    """
    split = patch_id[:1]
    if split == "T":
        return "train/valid"
    elif split == "E":
        return "test"

    raise ValueError(
        f"Unexpected split '{split}' in {filename}"
    )


In [ ]:
train_dataset = FloodDataset(
    split="train"
)
valid_dataset = FloodDataset(
    split="valid"
)
test_dataset = FloodDataset(
    split="test"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE_TRAIN,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE_VALID,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

In [ ]:
sample = test_dataset[13]

print(f"Pair ID : {sample['pair_id']}")
print(f"Patch ID: {sample['patch_id']}")

print(f"Before shape : {sample['before'].shape}")
print(f"After shape  : {sample['after'].shape}")
print(f"Metadata     : {sample['metadata']}")
print(f"Z-score shape: {sample['zscore'].shape}")

CHANNEL_NAMES = [
    "SAR",
    "Terrain"
]

fig, axes = plt.subplots(
    2,
    3,
    figsize=(12, 8)
)

for i in range(2):

    # Before
    axes[0, i].imshow(
        sample["before"][i],
        cmap="gray"
    )
    axes[0, i].set_title(
        f"Before ({CHANNEL_NAMES[i]})"
    )
    axes[0, i].axis("off")

    # After
    axes[1, i].imshow(
        sample["after"][i],
        cmap="gray"
    )
    axes[1, i].set_title(
        f"After ({CHANNEL_NAMES[i]})"
    )
    axes[1, i].axis("off")

# Ground-truth Z-score
im = axes[0, 2].imshow(
    sample["zscore"][0],
    cmap="RdBu_r",
    vmin=-5,
    vmax=5
)
axes[0, 2].set_title("Ground Truth Z-score")
axes[0, 2].axis("off")

# Metadata
axes[1, 2].axis("off")

metadata_text = "\n".join([
    f"{name}: {value:.2f}"
    for name, value in zip(
        METADATA_COLUMNS + [
            "orbit",
            "look_side"
        ],
        sample["metadata"].numpy()
    )
])

axes[1, 2].text(
    0,
    1,
    metadata_text,
    fontsize=11,
    va="top",
    family="monospace"
)

fig.colorbar(
    im,
    ax=axes[0, 2],
    shrink=0.8,
    label="Z-score"
)

plt.tight_layout()

plt.show()

In [ ]:
batch = next(iter(train_loader))

print(batch["before"].shape)
print(batch["after"].shape)
print(batch["zscore"].shape)

In [ ]:
print(torch.isnan(batch["before"]).any())
print(torch.isnan(batch["after"]).any())
print(torch.isnan(batch["zscore"]).any())

In [ ]:
import copy
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

In [ ]:
DIR_CHECKPOINTS = Path("/content/drive/MyDrive/School/University-of-Guelph/MDS/Course/26-2S_DATA6700/2_model_checkpoints")
DIR_CHECKPOINTS.mkdir(
    parents=True,
    exist_ok=True
)
CHECKPOINT_NAME_BASE = "siamese_unet"

In [ ]:
class ResNet18Encoder(nn.Module):
    """
    ResNet18 encoder for Siamese U-Net.

    Input
    -----
    (B, 2, 256, 256)

    Output
    ------
    f1 : (B, 64,128,128)
    f2 : (B, 64, 64, 64)
    f3 : (B,128, 32, 32)
    f4 : (B,256, 16, 16)
    f5 : (B,512,  8,  8)
    """
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )
        # ----------------------------------------
        # Replace first convolution
        # ----------------------------------------
        old_conv = backbone.conv1
        backbone.conv1 = nn.Conv2d(
            INPUT_CHANNELS,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False
        )

        # ----------------------------------------
        # Initialize weights
        # ----------------------------------------
        with torch.no_grad():
            # Use pretrained SAR channel
            backbone.conv1.weight[:, 0] = old_conv.weight[:, 0]
            # Initialize terrain channel
            backbone.conv1.weight[:, 1] = (
                old_conv.weight.mean(dim=1)
            )
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

    def forward(self, x):
        # 64 ﷿﷿ 128 ﷿﷿ 128
        f1 = self.relu(
            self.bn1(
                self.conv1(x)
            )
        )
        # 64 ﷿﷿ 64 ﷿﷿ 64
        x = self.maxpool(f1)
        f2 = self.layer1(x)

        # 128 ﷿﷿ 32 ﷿﷿ 32
        f3 = self.layer2(f2)

        # 256 ﷿﷿ 16 ﷿﷿ 16
        f4 = self.layer3(f3)

        # 512 ﷿﷿ 8 ﷿﷿ 8
        f5 = self.layer4(f4)

        return (f1, f2, f3, f4, f5)


In [ ]:
# encoder = ResNet18Encoder().to(DEVICE)
# batch = next(iter(train_loader))
# before = batch["before"].to(DEVICE)
# features = encoder(before)

# for i, feature in enumerate(features, start=1):
#     print(f"f{i}: {feature.shape}")

# # f1: torch.Size([32, 64, 128, 128])
# # f2: torch.Size([32, 64, 64, 64])
# # f3: torch.Size([32, 128, 32, 32])
# # f4: torch.Size([32, 256, 16, 16])
# # f5: torch.Size([32, 512, 8, 8])

In [ ]:
class FeatureFusion(nn.Module):

    def forward(self, before, after):
        fused = []
        for fb, fa in zip(before, after):
            diff = torch.abs(fb - fa)
            fused.append(
                torch.cat(
                    [fb, fa, diff],
                    dim=1
                )
            )
        return fused

In [ ]:
# before = encoder(sample["before"])
# after = encoder(sample["after"])
# fusion = FeatureFusion()
# features = fusion(before, after)

# for f in features:
#     print(f.shape)

In [ ]:
class FiLM(nn.Module):
    """
    Feature-wise Linear Modulation (FiLM).

    Parameters
    ----------
    feature_dim : int
        Number of feature channels.
    metadata_dim : int
        Dimension of metadata vector.
    """

    def __init__(self, feature_dim, metadata_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(metadata_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, feature_dim * 2)
        )

    def forward(self, feature, metadata):
        """
        feature : (B,C,H,W)
        metadata: (B,metadata_dim)
        """
        film = self.mlp(metadata)
        gamma, beta = torch.chunk(
            film,
            chunks=2,
            dim=1
        )
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)
        beta = beta.unsqueeze(-1).unsqueeze(-1)

        return (1 + gamma) * feature + beta

In [ ]:
class DecoderBlock(nn.Module):
    """
    One decoder block of the Siamese U-Net.

    Structure
    ---------
    Upsample
        ﷿﷿﷿
    Concatenate skip connection
        ﷿﷿﷿
    Conv3﷿﷿3 + BN + ReLU
        ﷿﷿﷿
    Conv3﷿﷿3 + BN + ReLU
    """

    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(
            scale_factor=2,
            mode="bilinear",
            align_corners=False
        )
        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels + skip_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)

        return x

In [ ]:
# block = DecoderBlock(
#     in_channels=1536,
#     skip_channels=768,
#     out_channels=256
# ).to(DEVICE)
# x = torch.randn(2, 1536, 8, 8, device=DEVICE)
# skip = torch.randn(2, 768, 16, 16, device=DEVICE)
# y = block(x, skip)
# print(y.shape)
# # torch.Size([2, 256, 16, 16])

In [ ]:
class FeatureProjection(nn.Module):
    """
    Reduce the channel dimension of fused features
    before passing them to FiLM and the decoder.
    """

    def __init__(self):
        super().__init__()
        self.projections = nn.ModuleList([
            nn.Conv2d(192, 64, kernel_size=1, bias=False),
            nn.Conv2d(192, 64, kernel_size=1, bias=False),
            nn.Conv2d(384, 128, kernel_size=1, bias=False),
            nn.Conv2d(768, 256, kernel_size=1, bias=False),
            nn.Conv2d(1536, 512, kernel_size=1, bias=False),
        ])

    def forward(self, features):
        """
        Parameters
        ----------
        features : tuple (f1, f2, f3, f4, f5)

        Returns
        -------
        tuple
            Projected feature maps.
        """
        return tuple(
            proj(feature) for proj, feature in zip(
                self.projections, features
            )
        )

In [ ]:
# projection = FeatureProjection().to(DEVICE)

# batch = next(iter(train_loader))

# before = batch["before"].to(DEVICE)
# after = batch["after"].to(DEVICE)

# encoder = ResNet18Encoder().to(DEVICE)
# fusion = FeatureFusion()

# before_features = encoder(before)
# after_features = encoder(after)

# fused = fusion(before_features, after_features)
# projected = projection(fused)

# for feature in projected:
#     print(feature.shape)

In [ ]:
class SiameseUNet(nn.Module):
    """
    Siamese U-Net with

    - Shared ResNet18 encoder
    - Multi-scale feature fusion
    - Feature projection
    - FiLM conditioning
    - U-Net decoder
    """

    def __init__(self):
        super().__init__()

        # -----------------------------------------
        # Encoder
        # -----------------------------------------
        self.encoder = ResNet18Encoder()

        # -----------------------------------------
        # Feature fusion
        # -----------------------------------------
        self.fusion = FeatureFusion()

        # -----------------------------------------
        # Channel projection
        # -----------------------------------------
        self.projection = FeatureProjection()

        # -----------------------------------------
        # Metadata conditioning
        # -----------------------------------------
        self.film = FiLM(
            feature_dim=512,
            metadata_dim=METADATA_DIM
        )

        # -----------------------------------------
        # Decoder
        # -----------------------------------------
        self.decoder4 = DecoderBlock(
            in_channels=512,
            skip_channels=256,
            out_channels=256
        )
        self.decoder3 = DecoderBlock(
            in_channels=256,
            skip_channels=128,
            out_channels=128
        )
        self.decoder2 = DecoderBlock(
            in_channels=128,
            skip_channels=64,
            out_channels=64
        )
        self.decoder1 = DecoderBlock(
            in_channels=64,
            skip_channels=64,
            out_channels=64
        )

        # -----------------------------------------
        # Final upsampling
        # -----------------------------------------
        self.final = nn.Sequential(
            nn.Upsample(
                scale_factor=2,
                mode="bilinear",
                align_corners=False
            ),
            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1)
        )

    def forward(self, before, after, metadata):
        # ----------------------------
        # Shared encoder
        # ----------------------------
        before = self.encoder(before)
        after = self.encoder(after)

        # ----------------------------
        # Siamese fusion
        # ----------------------------
        features = self.fusion(before, after)

        # ----------------------------
        # Channel projection
        # ----------------------------
        features = self.projection(features)
        f1, f2, f3, f4, f5 = features

        # ----------------------------
        # FiLM
        # ----------------------------
        x = self.film(f5, metadata)

        # ----------------------------
        # Decoder
        # ----------------------------
        x = self.decoder4(x, f4)
        x = self.decoder3(x, f3)
        x = self.decoder2(x, f2)
        x = self.decoder1(x, f1)

        # ----------------------------
        # Prediction
        # ----------------------------
        x = self.final(x)

        return x

In [ ]:
# torch.cuda.empty_cache()

In [ ]:
# Initialize model
model = SiameseUNet().to(DEVICE)
print(model)

In [ ]:
# Test model
batch = next(iter(train_loader))

prediction = model(
    batch["before"].to(DEVICE),
    batch["after"].to(DEVICE),
    batch["metadata"].to(DEVICE)
)
print(prediction.shape)
# torch.Size([BATCH_SIZE, 1, 256, 256])

In [ ]:
criterion = nn.SmoothL1Loss(beta=1.0)
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [ ]:
import torch.nn.functional as F

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer
):
    model.train()

    running_loss = 0.0
    running_rmse = 0.0
    total = 0

    for batch in loader:
        before = batch["before"].to(DEVICE)
        after = batch["after"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE)
        zscore = batch["zscore"].to(DEVICE)

        optimizer.zero_grad()
        prediction = model(
            before,
            after,
            metadata
        )
        loss = criterion(
            prediction,
            zscore
        )

        loss.backward()
        optimizer.step()

        batch_size = before.size(0)

        running_loss += loss.item() * batch_size
        rmse = torch.sqrt(
            F.mse_loss(prediction, zscore)
        )
        running_rmse += rmse.item() * batch_size
        total += batch_size

    loss = running_loss / total
    rmse = running_rmse / total

    return loss, rmse

@torch.no_grad()
def validate(
    model,
    loader,
    criterion
):
    model.eval()

    running_loss = 0.0
    running_rmse = 0.0
    total = 0

    for batch in loader:
        before = batch["before"].to(DEVICE)
        after = batch["after"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE)
        zscore = batch["zscore"].to(DEVICE)

        prediction = model(
            before,
            after,
            metadata
        )

        loss = criterion(
            prediction,
            zscore
        )

        batch_size = before.size(0)

        running_loss += loss.item() * batch_size

        rmse = torch.sqrt(
            F.mse_loss(
                prediction,
                zscore
            )
        )

        running_rmse += rmse.item() * batch_size

        total += batch_size

    loss = running_loss / total
    rmse = running_rmse / total

    return loss, rmse

In [ ]:
best_loss = float("inf")
history = {
    "train_loss": [],
    "valid_loss": [],
    "train_rmse": [],
    "valid_rmse": []
}

start = time.time()

for epoch in range(EPOCHS):

    train_loss, train_rmse = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    valid_loss, valid_rmse = validate(
        model,
        valid_loader,
        criterion
    )

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["valid_loss"].append(valid_loss)
    history["train_rmse"].append(train_rmse)
    history["valid_rmse"].append(valid_rmse)

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train RMSE {train_rmse:.4f} | "
        f"Valid Loss {valid_loss:.4f} | "
        f"Valid RMSE {valid_rmse:.4f}"
    )

    if valid_loss < best_loss:

        best_loss = valid_loss

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        model_name = (
            f"{CHECKPOINT_NAME_BASE}_"
            f"{timestamp}_"
            f"epoch{epoch+1:02d}_"
            f"loss{best_loss:.4f}.pth"
        )

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "valid_loss": valid_loss,
                "valid_rmse": valid_rmse,
            },
            DIR_CHECKPOINTS / model_name
        )

        print("﷿﷿﷿ Best model updated.")

elapsed = time.time() - start

print(f"\nTraining finished in {elapsed/60:.1f} minutes.")
print(f"Best validation loss: {best_loss:.4f}")

Epoch 001 | Train Loss 0.2894 | Train RMSE 0.7860 | Valid Loss 0.2215 | Valid RMSE 0.6664
﷿﷿﷿ Best model updated.
Epoch 002 | Train Loss 0.2268 | Train RMSE 0.6820 | Valid Loss 0.2255 | Valid RMSE 0.6764
Epoch 003 | Train Loss 0.2125 | Train RMSE 0.6602 | Valid Loss 0.2901 | Valid RMSE 0.7431
Epoch 004 | Train Loss 0.2058 | Train RMSE 0.6439 | Valid Loss 0.1874 | Valid RMSE 0.5979
﷿﷿﷿ Best model updated.
Epoch 005 | Train Loss 0.1924 | Train RMSE 0.6239 | Valid Loss 0.1993 | Valid RMSE 0.6263
Epoch 006 | Train Loss 0.1882 | Train RMSE 0.6158 | Valid Loss 0.2390 | Valid RMSE 0.6797
Epoch 007 | Train Loss 0.1923 | Train RMSE 0.6206 | Valid Loss 0.2348 | Valid RMSE 0.6783
Epoch 008 | Train Loss 0.1829 | Train RMSE 0.6038 | Valid Loss 0.2672 | Valid RMSE 0.7193
Epoch 009 | Train Loss 0.1813 | Train RMSE 0.6020 | Valid Loss 0.2138 | Valid RMSE 0.6469
Epoch 010 | Train Loss 0.1647 | Train RMSE 0.5727 | Valid Loss 0.2264 | Valid RMSE 0.6609
Epoch 011 | Train Loss 0.1586 | Train RMSE 0.5631 | Valid Loss 0.1923 | Valid RMSE 0.6144
Epoch 012 | Train Loss 0.1609 | Train RMSE 0.5628 | Valid Loss 0.2562 | Valid RMSE 0.7030
Epoch 013 | Train Loss 0.1530 | Train RMSE 0.5523 | Valid Loss 0.1757 | Valid RMSE 0.5918
﷿﷿﷿ Best model updated.
Epoch 014 | Train Loss 0.1451 | Train RMSE 0.5376 | Valid Loss 0.1752 | Valid RMSE 0.5831
﷿﷿﷿ Best model updated.
Epoch 015 | Train Loss 0.1563 | Train RMSE 0.5540 | Valid Loss 0.2937 | Valid RMSE 0.7590
Epoch 016 | Train Loss 0.1399 | Train RMSE 0.5255 | Valid Loss 0.1364 | Valid RMSE 0.5152
﷿﷿﷿ Best model updated.
Epoch 017 | Train Loss 0.1440 | Train RMSE 0.5310 | Valid Loss 0.1969 | Valid RMSE 0.6158
Epoch 018 | Train Loss 0.1305 | Train RMSE 0.5067 | Valid Loss 0.1582 | Valid RMSE 0.5518
Epoch 019 | Train Loss 0.1364 | Train RMSE 0.5171 | Valid Loss 0.1671 | Valid RMSE 0.5680
Epoch 020 | Train Loss 0.1329 | Train RMSE 0.5105 | Valid Loss 0.1713 | Valid RMSE 0.5804
Epoch 021 | Train Loss 0.1354 | Train RMSE 0.5109 | Valid Loss 0.2311 | Valid RMSE 0.6732
Epoch 022 | Train Loss 0.1195 | Train RMSE 0.4856 | Valid Loss 0.2448 | Valid RMSE 0.6898
Epoch 023 | Train Loss 0.1177 | Train RMSE 0.4796 | Valid Loss 0.1181 | Valid RMSE 0.4807
﷿﷿﷿ Best model updated.
Epoch 024 | Train Loss 0.1214 | Train RMSE 0.4866 | Valid Loss 0.1429 | Valid RMSE 0.5274
Epoch 025 | Train Loss 0.1140 | Train RMSE 0.4732 | Valid Loss 0.1435 | Valid RMSE 0.5259
Epoch 026 | Train Loss 0.1113 | Train RMSE 0.4662 | Valid Loss 0.1251 | Valid RMSE 0.4908
Epoch 027 | Train Loss 0.1046 | Train RMSE 0.4534 | Valid Loss 0.1486 | Valid RMSE 0.5325
Epoch 028 | Train Loss 0.1041 | Train RMSE 0.4493 | Valid Loss 0.1488 | Valid RMSE 0.5318
Epoch 029 | Train Loss 0.0961 | Train RMSE 0.4337 | Valid Loss 0.1092 | Valid RMSE 0.4620
﷿﷿﷿ Best model updated.
Epoch 030 | Train Loss 0.0968 | Train RMSE 0.4362 | Valid Loss 0.1316 | Valid RMSE 0.5020
Epoch 031 | Train Loss 0.0931 | Train RMSE 0.4287 | Valid Loss 0.1356 | Valid RMSE 0.5101
Epoch 032 | Train Loss 0.0863 | Train RMSE 0.4120 | Valid Loss 0.1551 | Valid RMSE 0.5436
Epoch 033 | Train Loss 0.0845 | Train RMSE 0.4083 | Valid Loss 0.1496 | Valid RMSE 0.5287
Epoch 034 | Train Loss 0.0832 | Train RMSE 0.4040 | Valid Loss 0.1307 | Valid RMSE 0.4989
Epoch 035 | Train Loss 0.0842 | Train RMSE 0.4052 | Valid Loss 0.1144 | Valid RMSE 0.4702
Epoch 036 | Train Loss 0.0825 | Train RMSE 0.4013 | Valid Loss 0.1286 | Valid RMSE 0.4972
Epoch 037 | Train Loss 0.0775 | Train RMSE 0.3923 | Valid Loss 0.1228 | Valid RMSE 0.4848
Epoch 038 | Train Loss 0.0737 | Train RMSE 0.3808 | Valid Loss 0.1420 | Valid RMSE 0.5198
Epoch 039 | Train Loss 0.0723 | Train RMSE 0.3793 | Valid Loss 0.1135 | Valid RMSE 0.4691
Epoch 040 | Train Loss 0.0759 | Train RMSE 0.3842 | Valid Loss 0.1209 | Valid RMSE 0.4810
Epoch 041 | Train Loss 0.0748 | Train RMSE 0.3818 | Valid Loss 0.1115 | Valid RMSE 0.4647
Epoch 042 | Train Loss 0.0720 | Train RMSE 0.3766 | Valid Loss 0.1240 | Valid RMSE 0.4860
Epoch 043 | Train Loss 0.0748 | Train RMSE 0.3814 | Valid Loss 0.1667 | Valid RMSE 0.5629
Epoch 044 | Train Loss 0.0744 | Train RMSE 0.3817 | Valid Loss 0.1624 | Valid RMSE 0.5531
Epoch 045 | Train Loss 0.0713 | Train RMSE 0.3748 | Valid Loss 0.1336 | Valid RMSE 0.5038
Epoch 046 | Train Loss 0.0715 | Train RMSE 0.3740 | Valid Loss 0.1287 | Valid RMSE 0.4934
Epoch 047 | Train Loss 0.0692 | Train RMSE 0.3689 | Valid Loss 0.1461 | Valid RMSE 0.5269
Epoch 048 | Train Loss 0.0718 | Train RMSE 0.3749 | Valid Loss 0.1692 | Valid RMSE 0.5666
Epoch 049 | Train Loss 0.0689 | Train RMSE 0.3690 | Valid Loss 0.1298 | Valid RMSE 0.4950
Epoch 050 | Train Loss 0.0697 | Train RMSE 0.3701 | Valid Loss 0.1200 | Valid RMSE 0.4782

Training finished in 18.1 minutes.
Best validation loss: 0.1092


In [ ]:
model = SiameseUNet().to(DEVICE)
checkpoint = torch.load(
    DIR_CHECKPOINTS / "siamese_unet_20260713_215250_epoch29_loss0.1092.pth",
    map_location=DEVICE
)
model.load_state_dict(
    checkpoint["model_state_dict"]
)
model.eval()

In [ ]:
from scipy.stats import pearsonr
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
@torch.no_grad()
def evaluate_test(model, loader):
    model.eval()
    y_true = []
    y_pred = []

    for batch in loader:
        before = batch["before"].to(DEVICE)
        after = batch["after"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE)

        prediction = model(
            before,
            after,
            metadata
        )
        prediction = prediction.cpu().numpy()
        target = batch["zscore"].numpy()

        y_pred.append(prediction.reshape(-1))
        y_true.append(target.reshape(-1))

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    r2 = r2_score(y_true, y_pred)
    corr, _ = pearsonr(y_true, y_pred)

    print(f"MAE   : {mae:.4f}")
    print(f"RMSE  : {rmse:.4f}")
    print(f"R﷿﷿    : {r2:.4f}")
    print(f"Corr. : {corr:.4f}")

    return y_true, y_pred

In [ ]:
y_true, y_pred = evaluate_test(
    model,
    test_loader
)

MAE   : 0.3964
RMSE  : 0.5017
R﷿﷿    : 0.7684
Corr. : 0.8810

In [ ]:
from scipy.stats import pearsonr

@torch.no_grad()
def show_prediction(
    model,
    dataset,
    index,
    device=DEVICE
):
    """
    Visualize one prediction.

    Displays:
        - SAR Before
        - SAR After
        - Ground Truth Z-score
        - Predicted Z-score
        - Absolute Error
        - Signed Error (Prediction - Ground Truth)
    """

    model.eval()

    sample = dataset[index]

    before = sample["before"].unsqueeze(0).to(device)
    after = sample["after"].unsqueeze(0).to(device)
    metadata = sample["metadata"].unsqueeze(0).to(device)

    # --------------------------------------------------
    # Prediction
    # --------------------------------------------------
    prediction = model(
        before,
        after,
        metadata
    )

    prediction = prediction.squeeze().cpu().numpy()
    target = sample["zscore"].squeeze().numpy()

    # --------------------------------------------------
    # Inputs
    # --------------------------------------------------
    before_img = sample["before"][0].cpu().numpy()
    after_img = sample["after"][0].cpu().numpy()

    # --------------------------------------------------
    # Metrics
    # --------------------------------------------------
    error = np.abs(prediction - target)
    signed_error = prediction - target

    rmse = np.sqrt(
        np.mean(
            (prediction - target) ** 2
        )
    )

    corr, _ = pearsonr(
        prediction.ravel(),
        target.ravel()
    )

    # --------------------------------------------------
    # Common color limits
    # --------------------------------------------------
    vmin = min(
        prediction.min(),
        target.min()
    )

    vmax = max(
        prediction.max(),
        target.max()
    )

    # --------------------------------------------------
    # Plot
    # --------------------------------------------------
    fig, ax = plt.subplots(
        3,
        2,
        figsize=(12, 14)
    )

    # -----------------------------
    # Row 1
    # -----------------------------
    ax[0,0].imshow(
        before_img,
        cmap="gray"
    )
    ax[0,0].set_title("SAR Before")
    ax[0,0].axis("off")

    ax[0,1].imshow(
        after_img,
        cmap="gray"
    )
    ax[0,1].set_title("SAR After")
    ax[0,1].axis("off")

    # -----------------------------
    # Row 2
    # -----------------------------
    im = ax[1,0].imshow(
        target,
        cmap="RdBu_r",
        vmin=vmin,
        vmax=vmax
    )
    ax[1,0].set_title("Ground Truth")
    ax[1,0].axis("off")

    ax[1,1].imshow(
        prediction,
        cmap="RdBu_r",
        vmin=vmin,
        vmax=vmax
    )
    ax[1,1].set_title("Prediction")
    ax[1,1].axis("off")

    # -----------------------------
    # Row 3
    # -----------------------------
    im_err = ax[2,0].imshow(
        error,
        cmap="magma"
    )
    ax[2,0].set_title("Absolute Error")
    ax[2,0].axis("off")

    err_lim = np.max(
        np.abs(signed_error)
    )

    im_diff = ax[2,1].imshow(
        signed_error,
        cmap="RdBu_r",
        vmin=-err_lim,
        vmax=err_lim
    )
    ax[2,1].set_title("Prediction - Ground Truth")
    ax[2,1].axis("off")

    # --------------------------------------------------
    # Colorbars
    # --------------------------------------------------
    fig.colorbar(
        im,
        ax=ax[1,:],
        shrink=0.8,
        label="Z-score"
    )

    fig.colorbar(
        im_err,
        ax=ax[2,0],
        shrink=0.8,
        label="Absolute Error"
    )

    fig.colorbar(
        im_diff,
        ax=ax[2,1],
        shrink=0.8,
        label="Signed Error"
    )

    # --------------------------------------------------
    # Title
    # --------------------------------------------------
    fig.suptitle(
        f"Pair : {sample['pair_id']}\n"
        f"Patch: {sample['patch_id']}\n"
        f"RMSE = {rmse:.3f}    "
        f"Correlation = {corr:.3f}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

In [ ]:
show_prediction(
    model,
    test_dataset,
    index=7
)

In [ ]:
from pathlib import Path
import numpy as np
import rasterio

In [ ]:
@torch.no_grad()
def export_prediction_geotiffs(
    model,
    dataset,
    output_dir,
    device=DEVICE
):
    """
    Export predicted Z-score maps as GeoTIFFs.

    Output filename:
        prediction_{pair_id}_{patch_id}.tif
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    model.eval()
    for idx in range(len(dataset)):
        tensor_sample = dataset[idx]
        file_sample = dataset.samples[idx]

        before = (
            tensor_sample["before"]
            .unsqueeze(0)
            .to(device)
        )
        after = (
            tensor_sample["after"]
            .unsqueeze(0)
            .to(device)
        )
        metadata = (
            tensor_sample["metadata"]
            .unsqueeze(0)
            .to(device)
        )

        # -----------------------------------------
        # Prediction
        # -----------------------------------------
        prediction = model(
            before,
            after,
            metadata
        )
        prediction = (
            prediction
            .squeeze()
            .cpu()
            .numpy()
            .astype(np.float32)
        )
        groundtruth = (
            tensor_sample["zscore"]
            .squeeze()
            .cpu()
            .numpy()
            .astype(np.float32)
        )
        error = prediction - groundtruth
        rmse = np.sqrt(np.mean(error ** 2))

        # -----------------------------------------
        # Save GeoTIFF
        # -----------------------------------------
        with rasterio.open(file_sample["zscore"]) as src:
            profile = src.profile.copy()
            profile.update(
                driver="GTiff",
                dtype="float32",
                count=1,
                compress="LZW"
            )
            output_path = (
                output_dir /
                "patches_prediction" /
                (
                    f"prediction_"
                    f"{tensor_sample['pair_id']}_"
                    f"{tensor_sample['patch_id']}_"
                    f"rmse{rmse:.4f}.tif"
                )
            )
            with rasterio.open(output_path, "w", **profile) as dst:
                dst.write(prediction, 1)

            error_path = (
                output_dir /
                "patches_error" /
                (
                    f"error_"
                    f"{tensor_sample['pair_id']}_"
                    f"{tensor_sample['patch_id']}_"
                    f"rmse{rmse:.4f}.tif"
                )
            )
            with rasterio.open(error_path, "w", **profile) as dst:
                  dst.write(error, 1)

        print(
            f"[{idx+1:3d}/{len(dataset)}] "
            f"{output_path.name} | "
            f"{error_path.name}"
        )

    print("Done!")

In [ ]:
export_prediction_geotiffs(
    model,
    test_dataset,
    DIR_RESULTS
)